# PV & Wind Power Curve Modeling

Erick Chauke

Physics-informed and data-driven power curve models for two PV plants and two wind plants, fit
from source power-curve grids (input files, gitignored). Six standalone models: PV1, PV2,
PV-combined, Wind1, Wind2, Wind-combined.

## Setup

Imports and one config cell for the source path and adjustable parameters.

### Imports

Numeric, tabular, plotting, and Excel-reading libraries used throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl

### Configuration

File path and adjustable parameters used by every later section.

In [ ]:
# Config: file path and adjustable params. Capacity values sourced from each sheet's
# title cell (row 0, col 0), e.g. "Installed Capacity: 75 MW".
DATA_PATH = "data/FPCs.xlsm"

PV_IRRADIANCE_STEP = 50    # W/m^2
PV_TEMP_STEP = 5           # deg C
WIND_VELOCITY_STEP = 0.5   # m/s
WIND_DIRECTION_STEP = 15   # deg

CV_FOLDS = 5

CAPACITY_MW = {
    "PV1": 75.0,
    "PV2": 75.0,
    "Wind1": 102.0,
    "Wind2": 86.6,
}

## Data Loading & Grid Parsing

Each sheet is a 2D power-curve grid, parsed by position (not pandas' auto-detected headers, which the merged/label cells break).

### Grid parsing helper

Positionally slices a sheet into capacity label, x-axis, y-axis, and power grid.

In [ ]:
def parse_grid_sheet(path, sheet_name):
    """Positional parse of one FPCs.xlsm-style grid sheet into title, x-axis, y-axis, power grid."""
    df = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="openpyxl")
    capacity_label = df.iloc[0, 0]
    x_axis = df.iloc[1, 2:].astype(float).to_numpy()
    y_axis = df.iloc[2:, 1].astype(float).to_numpy()
    power_grid = df.iloc[2:, 2:].astype(float).to_numpy()
    return capacity_label, x_axis, y_axis, power_grid

### Load all four sheets

Applies the helper to PV1, PV2, Wind1, and Wind2 into one dict keyed by sheet name.

In [ ]:
grids = {}
for sheet_name in ["PV1", "PV2", "Wind1", "Wind2"]:
    capacity_label, x_axis, y_axis, power_grid = parse_grid_sheet(DATA_PATH, sheet_name)
    grids[sheet_name] = {
        "capacity_label": capacity_label,
        "x_axis": x_axis,
        "y_axis": y_axis,
        "power_grid": power_grid,
    }

### Sanity check

Confirms shapes and axis ranges for all four parsed grids.

In [ ]:
for sheet_name, g in grids.items():
    print(
        sheet_name, "|", g["capacity_label"],
        "| grid shape:", g["power_grid"].shape,
        "| x range:", g["x_axis"].min(), "-", g["x_axis"].max(),
        "| y range:", g["y_axis"].min(), "-", g["y_axis"].max(),
    )

## Exploratory Data Analysis

Raw grid heatmaps, the PV missingness pattern, and the Wind2 negative-value artifact.

### Raw grid heatmaps

Power grid for each of the four sheets, plotted over its native axes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, sheet_name in zip(axes.flat, ["PV1", "PV2", "Wind1", "Wind2"]):
    g = grids[sheet_name]
    is_pv = sheet_name.startswith("PV")
    im = ax.imshow(
        g["power_grid"], origin="lower", aspect="auto",
        extent=[g["x_axis"].min(), g["x_axis"].max(), g["y_axis"].min(), g["y_axis"].max()],
    )
    ax.set_title(f"{sheet_name} ({g['capacity_label']})")
    ax.set_xlabel("Module Temperature (C)" if is_pv else "Wind Direction (deg)")
    ax.set_ylabel("Irradiance (W/m^2)" if is_pv else "Wind Velocity (m/s)")
    fig.colorbar(im, ax=ax, label="Power (MW)")
fig.tight_layout()
fig.savefig("outputs/all_sheets_raw_grids.png", dpi=150)
plt.show()

### PV missingness pattern

Fill rate for PV1 and PV2, checked against the documented values.

In [ ]:
for sheet_name in ["PV1", "PV2"]:
    power_grid = grids[sheet_name]["power_grid"]
    fill_rate = 100 * (1 - np.isnan(power_grid).mean())
    print(f"{sheet_name}: {fill_rate:.1f}% filled ({np.isnan(power_grid).sum()} missing cells)")

### PV missingness heatmap

Where the missing cells fall in (irradiance, module temperature) space, for PV1 and PV2 side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, sheet_name in zip(axes, ["PV1", "PV2"]):
    g = grids[sheet_name]
    missing_mask = np.isnan(g["power_grid"])
    ax.imshow(
        missing_mask, origin="lower", aspect="auto", cmap="gray_r",
        extent=[g["x_axis"].min(), g["x_axis"].max(), g["y_axis"].min(), g["y_axis"].max()],
    )
    ax.set_title(f"{sheet_name} missing cells (black = missing)")
    ax.set_xlabel("Module Temperature (C)")
    ax.set_ylabel("Irradiance (W/m^2)")
fig.tight_layout()
fig.savefig("outputs/pv1_pv2_missingness.png", dpi=150)
plt.show()

### Wind2 negative-value check

Locate the documented near-zero negative artifact before deciding how to handle it.

In [ ]:
wind2_grid = grids["Wind2"]["power_grid"]
wind2_velocity = grids["Wind2"]["y_axis"]
wind2_direction = grids["Wind2"]["x_axis"]

for row, col in np.argwhere(wind2_grid < 0):
    print(
        f"velocity={wind2_velocity[row]} m/s, direction={wind2_direction[col]} deg, "
        f"power={wind2_grid[row, col]:.4f} MW"
    )

### Clip Wind2 artifact

Clip the negative cell(s) to 0 in place; this is measurement noise around zero wind speed, not real negative generation.

In [ ]:
# Clip near-zero negative artifact (measurement noise around zero wind speed, not real
# negative generation) -- see the Wind2 negative-value check above for the flagged cell(s).
grids["Wind2"]["power_grid"] = np.clip(wind2_grid, 0.0, None)
print("remaining negative cells:", (grids["Wind2"]["power_grid"] < 0).sum())

### Discussion

The raw grids look physically sane: PV1/PV2 power rises with irradiance and eases off with
higher module temperature at fixed irradiance, and both wind plants show the expected S-curve,
cutting in near 3 m/s and flattening at rated capacity (102 MW / 86.6 MW) by around 10-12 m/s,
with no strong dependence on wind direction.

PV1 is 53.2% filled (262 of 560 cells missing), PV2 is 47.9% filled (292 of 560), matching the
documented fill rates. The missingness heatmap shows why: filled cells form a diagonal band
running from (low irradiance, low temperature) to (high irradiance, high temperature). The
missing corners, cold module with high irradiance and hot module with little irradiance, are
combinations that essentially never occur in the field because irradiance is what heats the
module. This is a sampling artifact of physically coupled inputs, not a data quality problem, so
it is left as unsampled space for the models to generalize into rather than something to impute.

The Wind2 check found two near-zero negative cells (-0.095 MW and -0.057 MW), both at 2.5 m/s
(below cut-in), at wind directions 240 and 255 degrees, not the single cell the source
description mentioned. Both are measurement noise around zero, not real negative generation, and
have been clipped to 0 in `grids["Wind2"]["power_grid"]`.